# Atelier 2 — FASTERRCNN — adaptation KARMAS Hammam

Source : [Eachr4f/atelier2-mnist](https://github.com/Eachr4f/atelier2-mnist),
commit `47654402f475fbe0e1d608b8e8417f28bb080d27`. Les originaux exécutés sont conservés dans
[`references/eachr4f/notebooks/`](../../references/eachr4f/notebooks/).

**Cette version est corrigée et n'a pas encore été exécutée sur GPU.** Ses sorties
ont été effacées pour ne pas attribuer les anciens scores au code modifié.
Dans Colab : activer un GPU puis **Tout exécuter**. Un ou deux GPU sont acceptés.

Adaptations : chemins portables et téléchargement vérifié ; partage commun
54 000 train / 6 000 validation / 10 000 test (seed 42) ; export des nouveaux résultats.
Faster R-CNN : BatchNorm figées en validation, absence de détection = -1 et loss test
explicitement diagnostique. ViT : vraie validation, checkpoint sélectionné sur
validation et moyenne de loss test pondérée par les effectifs des lots.

Les architectures et réglages de base proviennent du dépôt attribué, avec une
convolution CNN à trois blocs, des têtes compactes pour VGG16/AlexNet et le ViT
pédagogique à 8 dimensions. Le partage et les corrections changent le protocole :
**ne pas réutiliser les scores d'origine comme résultats de cette adaptation.**

In [ ]:

from pathlib import Path
import os, sys, subprocess, importlib.util
ROOT = Path.cwd()
if not (ROOT / 'src/adaptation.py').exists():
    # Accepter un lancement depuis notebooks/colab/ ou depuis Colab lui-même.
    for candidate in list(ROOT.parents):
        if (candidate / 'src/adaptation.py').exists():
            ROOT = candidate
            break
    else:
        ROOT = Path('/content/Atelier2_DeepLearning') if Path('/content').exists() else Path('/kaggle/working/Atelier2_DeepLearning')
        if not ROOT.exists():
            subprocess.run(['git', 'clone', 'https://github.com/KarmasHammam/Atelier2_DeepLearning.git', str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = str(ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
if importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Active un GPU : Colab > Exécution > Modifier le type d’exécution > GPU.')
from src.adaptation import prepare_work, save_result
WORK_DIR = prepare_work('fasterrcnn')
os.environ['ATELIER2_WORK_DIR'] = str(WORK_DIR)
print('Adaptation KARMAS Hammam ; source Eachr4f, commit 47654402f475fbe0e1d608b8e8417f28bb080d27')
print('Matériel de cette session :', torch.cuda.get_device_name(0))
print('Les résultats de référence importés ne sont pas ceux de cette nouvelle exécution.')


# Lab 2 — Faster R-CNN sur MNIST

Notebook Kaggle avec GPU T4. MNIST est un jeu de classification et ne contient pas de boîtes de détection. Pour appliquer Faster R-CNN, ce notebook transforme chaque image en un exemple de détection d’un seul objet : le chiffre est la classe, et sa boîte englobante est calculée depuis les pixels non blancs. Cette adaptation permet l’exercice, mais les scores ne sont pas directement équivalents aux métriques du CNN classifier.

Activer un GPU dans Colab puis exécuter les cellules dans l’ordre. Le téléchargement MNIST est automatique.


In [ ]:
from pathlib import Path
import os
import gzip, shutil, random, copy, time, json, subprocess, sys
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("GPU indisponible. Kaggle > Settings > Accelerator > GPU T4 x2")
NUM_GPUS = torch.cuda.device_count()
print("GPU détectés :", NUM_GPUS)
for i in range(NUM_GPUS):
    print(f"GPU {i} : {torch.cuda.get_device_name(i)}")
print("DDP utilisera un ou deux GPU selon cette session.")


## Préparer les fichiers MNIST

Recherche des fichiers IDX dans `/kaggle/input`, compressés ou non.


In [ ]:
# Les IDX MNIST ont été téléchargés et vérifiés par prepare_work.
print('Données locales :', WORK_DIR / 'MNIST/raw')

## Créer les exemples de détection

Les images sont agrandies à 96×96 pour convenir aux cartes de caractéristiques du détecteur. Les classes Faster R-CNN sont `1` à `10` ; `0` reste la classe de fond.


In [ ]:
class MNISTDetection(Dataset):
    def __init__(self, train=True, indices=None):
        self.ds=datasets.MNIST(str(WORK_DIR / ''),train=train,download=False)
        self.indices=list(range(len(self.ds))) if indices is None else list(indices)
        self.resize=transforms.Resize((64,64),antialias=True)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        image,label=self.ds[self.indices[i]]
        gray=transforms.ToTensor()(image)
        rgb=self.resize(gray.repeat(3,1,1))
        ys,xs=torch.where(gray[0]>0.05)
        x1=max(0,int(xs.min())-1); y1=max(0,int(ys.min())-1)
        x2=min(28,int(xs.max())+2); y2=min(28,int(ys.max())+2)
        scale=64/28
        box=torch.tensor([[x1*scale,y1*scale,x2*scale,y2*scale]],dtype=torch.float32)
        target={'boxes':box,'labels':torch.tensor([int(label)+1],dtype=torch.int64),
                'image_id':torch.tensor([self.indices[i]],dtype=torch.int64),
                'area':torch.tensor([(x2-x1)*(y2-y1)*scale*scale],dtype=torch.float32),
                'iscrowd':torch.zeros((1,),dtype=torch.int64)}
        return rgb,target,int(label)

def collate_fn(batch):
    images,targets,digits=zip(*batch)
    return list(images),list(targets),torch.tensor(digits)

# Valeurs modifiables : commencez petit pour valider le pipeline, puis augmentez.
TRAIN_SIZE = 54000  
VAL_SIZE = 6000
BATCH_PER_GPU = 8
EPOCHS = 8
PATIENCE = 3

gen=torch.Generator().manual_seed(SEED)
perm=torch.randperm(60000,generator=gen).tolist()
train_idx=perm[VAL_SIZE:VAL_SIZE+TRAIN_SIZE]
val_idx=perm[:VAL_SIZE]
train_ds=MNISTDetection(train=True,indices=train_idx)
val_ds=MNISTDetection(train=True,indices=val_idx)
test_ds=MNISTDetection(train=False)
train_loader=DataLoader(train_ds,batch_size=8,shuffle=True,num_workers=2,pin_memory=True,collate_fn=collate_fn)
val_loader=DataLoader(val_ds,batch_size=8,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)
test_loader=DataLoader(test_ds,batch_size=8,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)
print(f'Train {len(train_ds)} | validation {len(val_ds)} | test {len(test_ds)}')


In [ ]:
# Montrer un échantillon de 8 images et leur boîte réelle
images,targets,digits=next(iter(train_loader))
fig,axs=plt.subplots(2,4,figsize=(10,5))
for ax,img,t,digit in zip(axs.flat,images[:8],targets[:8],digits[:8]):
    ax.imshow(img[0].numpy(),cmap='gray')
    b=t['boxes'][0]
    ax.add_patch(plt.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],fill=False,color='lime',linewidth=1.5))
    ax.set_title(f'Chiffre réel : {digit.item()}'); ax.axis('off')
plt.suptitle('MNIST converti en détection (boîtes réelles en vert)'); plt.tight_layout(); plt.show()


## Construire Faster R-CNN

Le modèle ResNet-50-FPN est initialisé sans poids pré-entraînés afin d’éviter tout téléchargement externe. Le détecteur apprend donc à la fois les caractéristiques et les régions d’intérêt sur MNIST.


In [ ]:
NUM_CLASSES=11  # classe 0 = fond, classes 1..10 = chiffres 0..9
print(f"Faster R-CNN ResNet50-FPN | taille image 64x64 | batch global visé {BATCH_PER_GPU * max(NUM_GPUS, 1)}")
print("Entraînement distribué PyTorch DDP : un processus par GPU.")


## Entraînement et validation

La loss Faster R-CNN est la somme des pertes de classification et de régression de boîtes (avec pertes RPN). Pour obtenir une accuracy comparable à une tâche de classification, on prend la classe de la détection ayant le score le plus élevé dans chaque image.


In [ ]:
# PyTorch DistributedDataParallel : un processus sur chacun des deux GPU Kaggle.
# Le script autonome évite les problèmes de démarrage multiprocessing dans Jupyter.
worker_script = r"""from src.engine import detection_loss
import os
from pathlib import Path
WORK_DIR=Path(os.environ["ATELIER2_WORK_DIR"])

import os, json, copy, time, random
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torchvision import datasets, transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn

SEED=42
class MNISTDetection(Dataset):
    def __init__(self, train=True, indices=None):
        self.ds=datasets.MNIST(str(WORK_DIR / ''),train=train,download=False)
        self.indices=list(range(len(self.ds))) if indices is None else list(indices)
        self.resize=transforms.Resize((64,64),antialias=True)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        image,label=self.ds[self.indices[i]]
        gray=transforms.ToTensor()(image)
        rgb=self.resize(gray.repeat(3,1,1))
        ys,xs=torch.where(gray[0]>0.05)
        x1=max(0,int(xs.min())-1); y1=max(0,int(ys.min())-1)
        x2=min(28,int(xs.max())+2); y2=min(28,int(ys.max())+2)
        scale=64/28
        box=torch.tensor([[x1*scale,y1*scale,x2*scale,y2*scale]],dtype=torch.float32)
        target={'boxes':box,'labels':torch.tensor([int(label)+1],dtype=torch.int64),
                'image_id':torch.tensor([self.indices[i]],dtype=torch.int64),
                'area':torch.tensor([(x2-x1)*(y2-y1)*scale*scale],dtype=torch.float32),
                'iscrowd':torch.zeros((1,),dtype=torch.int64)}
        return rgb,target,int(label)
def collate_fn(batch):
    images,targets,digits=zip(*batch)
    return list(images),list(targets),torch.tensor(digits)
def move_targets(ts,device):
    return [{k:v.to(device) for k,v in t.items()} for t in ts]

def main():
    dist.init_process_group(backend='nccl')
    rank=int(os.environ['LOCAL_RANK']); world=int(os.environ['WORLD_SIZE'])
    torch.cuda.set_device(rank); device=torch.device('cuda',rank)
    random.seed(SEED+rank); np.random.seed(SEED+rank); torch.manual_seed(SEED+rank)
    train_size=int(os.environ['TRAIN_SIZE']); val_size=int(os.environ['VAL_SIZE'])
    epochs=int(os.environ['EPOCHS']); patience=int(os.environ['PATIENCE'])
    bpg=int(os.environ['BATCH_PER_GPU'])
    g=torch.Generator().manual_seed(SEED)
    perm=torch.randperm(60000,generator=g).tolist()
    train_idx=perm[val_size:val_size+train_size]; val_idx=perm[:val_size]
    train_ds=MNISTDetection(True,train_idx); val_ds=MNISTDetection(True,val_idx)
    sampler=DistributedSampler(train_ds,num_replicas=world,rank=rank,shuffle=True,seed=SEED,drop_last=True)
    train_loader=DataLoader(train_ds,batch_size=bpg,sampler=sampler,num_workers=2,pin_memory=True,collate_fn=collate_fn,persistent_workers=True)
    # Same validation partition on each process; rank 0 evaluates, then broadcasts loss.
    val_loader=DataLoader(val_ds,batch_size=bpg,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)
    model=fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None,num_classes=11,min_size=64,max_size=64,
        rpn_pre_nms_top_n_train=300,rpn_post_nms_top_n_train=100,
        rpn_pre_nms_top_n_test=100,rpn_post_nms_top_n_test=50,box_detections_per_img=10).to(device)
    model=DDP(model,device_ids=[rank],output_device=rank,find_unused_parameters=False)
    optimizer=torch.optim.SGD(model.parameters(),lr=0.005,momentum=0.9,weight_decay=0.0005)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=1)
    best=float('inf'); best_state=None; stale=0; history=[]; start=time.time()
    for epoch in range(1,epochs+1):
        model.train(); sampler.set_epoch(epoch); run_loss=0.; count=0
        for images,targets,_ in train_loader:
            images=[x.to(device,non_blocking=True) for x in images]
            targets=move_targets(targets,device)
            optimizer.zero_grad(set_to_none=True)
            losses=model(images,targets); loss=sum(losses.values())
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); optimizer.step()
            run_loss+=loss.item()*len(images); count+=len(images)
        # Average epoch train loss across ranks.
        stats=torch.tensor([run_loss,count],device=device,dtype=torch.float64)
        dist.all_reduce(stats,op=dist.ReduceOp.SUM)
        train_loss=(stats[0]/stats[1]).item()
        # Validation only on rank 0, broadcast result so all ranks stop together.
        val_tensor=torch.zeros(1,device=device)
        if rank==0:
            model.module.train()
            vtotal=0.; vn=0
            with torch.no_grad():
                for images,targets,_ in val_loader:
                    images=[x.to(device,non_blocking=True) for x in images]
                    value=detection_loss(model.module,images,move_targets(targets,device),device)
                    vtotal+=value*len(images); vn+=len(images)
            val_tensor[0]=vtotal/vn
        dist.broadcast(val_tensor,src=0); val_loss=val_tensor.item(); scheduler.step(val_loss)
        if rank==0:
            history.append({'epoch':epoch,'train_loss':train_loss,'val_loss':val_loss})
            print(f'Epoch {epoch:02d}/{epochs} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | elapsed {(time.time()-start)/60:.1f} min',flush=True)
            if val_loss<best:
                best=val_loss; best_state=copy.deepcopy(model.module.state_dict()); stale=0
                torch.save(best_state,str(WORK_DIR / 'mnist_faster_rcnn_best.pth'))
            else: stale+=1
            if stale>=patience: print('Early stopping.',flush=True)
        stop=torch.tensor([1 if rank==0 and stale>=patience else 0],device=device)
        dist.broadcast(stop,src=0)
        if stop.item(): break
    if rank==0:
        torch.cuda.synchronize()
        with open(str(WORK_DIR / 'timing.json'), 'w') as timing_file:
            json.dump({'training_seconds': time.time()-start}, timing_file)
        with open(str(WORK_DIR / 'mnist_faster_rcnn_history.json'),'w') as f: json.dump(history,f)
        print(f'Temps total entraînement : {(time.time()-start)/60:.1f} minutes',flush=True)
    dist.barrier(); dist.destroy_process_group()
if __name__=='__main__': main()
"""
script_path=str(WORK_DIR / 'train_faster_rcnn_ddp.py')
with open(script_path,'w',encoding='utf-8') as f: f.write(worker_script)
if NUM_GPUS < 1: raise RuntimeError("Un GPU au moins est nécessaire.")
env=os.environ.copy()
env.update({'TRAIN_SIZE':str(TRAIN_SIZE),'VAL_SIZE':str(VAL_SIZE),'EPOCHS':str(EPOCHS),
            'PATIENCE':str(PATIENCE),'BATCH_PER_GPU':str(BATCH_PER_GPU)})
cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nnodes=1',f'--nproc_per_node={min(NUM_GPUS, 2)}',script_path]
print('Lancement DDP :', min(NUM_GPUS, 2), 'GPU')
subprocess.run(cmd,env=env,check=True)
with open(str(WORK_DIR / 'mnist_faster_rcnn_history.json'),encoding='utf-8') as f:
    history=json.load(f)
training_seconds=json.loads((WORK_DIR / 'timing.json').read_text())['training_seconds']
# Charger le modèle entraîné pour les cellules d'évaluation/affichage ci-dessous.
device=torch.device('cuda:0')
model=fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None,num_classes=11,min_size=64,max_size=64,
    rpn_pre_nms_top_n_train=300,rpn_post_nms_top_n_train=100,
    rpn_pre_nms_top_n_test=100,rpn_post_nms_top_n_test=50,box_detections_per_img=10).to(device)
model.load_state_dict(torch.load(str(WORK_DIR / 'mnist_faster_rcnn_best.pth'),map_location=device))
model.eval()


In [ ]:
# Courbes de loss train/validation
epochs=[r['epoch'] for r in history]
plt.figure(figsize=(7,4))
plt.plot(epochs,[r['train_loss'] for r in history],marker='o',label='Train')
plt.plot(epochs,[r['val_loss'] for r in history],marker='o',label='Validation')
plt.xlabel('Époque'); plt.ylabel('Loss Faster R-CNN'); plt.title('Loss par époque')
plt.grid(True,alpha=.3); plt.legend(); plt.tight_layout(); plt.show()


## Évaluer sur le test

La loss test est calculée en mode entraînement sans mise à jour des poids. L’accuracy et le F1 macro utilisent la classe de la détection au score maximal. La qualité des boîtes nécessiterait en plus une mesure de détection telle que mAP.


In [ ]:
# Prédiction sans les boîtes vraies ; loss diagnostique avec BatchNorm figées.
from sklearn.metrics import accuracy_score, f1_score, classification_report
from src.engine import detection_loss, image_prediction
model.eval()
y_true, y_pred = [], []
loss_sum, number = 0., 0
with torch.no_grad():
    for images, targets, digits in test_loader:
        images = [image.to(device) for image in images]
        moved = [{key: value.to(device) for key, value in target.items()} for target in targets]
        outputs = model(images)
        for output, digit in zip(outputs, digits):
            predicted, confidence, box = image_prediction(output)
            y_pred.append(predicted)  # -1 si aucune détection : jamais assimilé au chiffre zéro.
            y_true.append(int(digit))
        loss_sum += detection_loss(model, images, moved, device) * len(images)
        number += len(images)
test_loss = loss_sum / number
print('Accuracy :', accuracy_score(y_true, y_pred))
print('F1 macro :', f1_score(y_true, y_pred, labels=list(range(10)), average='macro', zero_division=0))
print('Loss diagnostique détection :', test_loss, '(non comparable à une cross-entropy seule)')
print(classification_report(y_true, y_pred, labels=list(range(10)), digits=4, zero_division=0))

In [ ]:
# Afficher quatre images test avec boîte prédite, chiffre prédit et chiffre réel
images,targets,digits=next(iter(test_loader))
with torch.no_grad():
    outputs=model([im.to(device) for im in images[:4]])
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,img,target,digit,out in zip(axs,images[:4],targets[:4],digits[:4],outputs):
    ax.imshow(img[0].numpy(),cmap='gray')
    gt=int(digit.item())
    if len(out['labels']):
        k=int(out['scores'].argmax()); pred=int(out['labels'][k].item())-1
        box=out['boxes'][k].cpu(); score=float(out['scores'][k].cpu())
        ax.add_patch(plt.Rectangle((box[0],box[1]),box[2]-box[0],box[3]-box[1],fill=False,color='red',linewidth=1.5))
        title=f'Prédit : {pred} ({score:.2f})\nRéel : {gt}'
    else:
        title=f'Aucune détection\nRéel : {gt}'
    ax.set_title(title,color='green' if len(out['labels']) and pred==gt else 'red'); ax.axis('off')
plt.suptitle('Quatre prédictions test (boîte prédite en rouge)'); plt.tight_layout(); plt.show()

torch.save(model.state_dict(),str(WORK_DIR / 'mnist_faster_rcnn_best.pth'))
print('Modèle sauvegardé : /kaggle/working/mnist_faster_rcnn_best.pth')


## Remarques pour le compte rendu

- Faster R-CNN renvoie une loss de détection composée de plusieurs termes ; elle n’a pas la même définition que la cross-entropy du CNN classifier.
- L’accuracy/F1 ci-dessus sont une adaptation en prenant la meilleure détection de chaque image. Elles aident à comparer la reconnaissance du chiffre, mais ne mesurent pas la précision de localisation.
- Pour comparer la détection au sens strict, rapporter aussi mAP avec une implémentation de métriques de détection et décrire la règle de boîte utilisée.
- La version utilise DDP si Kaggle propose réellement deux T4. TRAIN_SIZE=6000, VAL_SIZE=1000 et des images 64×64 donnent un démarrage raisonnable ; augmentez ces valeurs après avoir vérifié le temps par époque. L’early stopping suit la validation.


## Enregistrer cette nouvelle exécution

Les métriques sont calculées depuis les prédictions de cette session GPU. Le dossier
`results/adapted_gpu/` reste distinct des résultats importés et des anciens profils.
Enregistrer aussi le notebook dans GitHub **avec ses sorties**. L'archive permet de
conserver les résultats, même si les cinq modèles ne sont pas tous terminés.

In [ ]:
result = save_result('fasterrcnn', y_true, y_pred, test_loss, training_seconds, WORK_DIR)
from scripts.export_adapted_results import export
archive = export(allow_partial=True)
print('Archive :', archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Archive disponible localement.')